# Self-supervised audio (wav2vec, HuBERT) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float); return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def hz_to_mel(f): return 2595*np.log10(1+np.asarray(f)/700)
def mel_to_hz(m): return 700*(10**(np.asarray(m)/2595)-1)
def stft_mag(x,N=256,H=64):
    w=np.hanning(N); frames=[]
    for s in range(0,len(x)-N+1,H): frames.append(np.abs(np.fft.rfft(x[s:s+N]*w)))
    return np.array(frames).T
print('setup ok')

## Learning audio representations from hidden or contrasted frames
Self-supervised audio creates its own labels. These examples show contrastive probability, temperature, masking, cluster prediction, and representation separation.

In [ ]:
# 1) contrastive loss rewards the true latent over a distractor
c=np.array([1.,0.]); pos=np.array([0.8,0.6]); neg=np.array([0.1,0.995]); sims=np.array([cosine(c,pos),cosine(c,neg)]); p=softmax(sims/0.1); loss=-math.log(p[0])
plt.figure(figsize=(6,3)); plt.bar(['positive','negative'],p); plt.ylim(0,1); plt.title(f'positive prob={p[0]:.4f}, loss={loss:.4f}')
assert abs(p[0]-0.9990889601839508)<1e-9 and loss<0.001

In [ ]:
# 2) temperature sharpens or softens the contrastive choice
sims=np.array([0.8,0.1]); temps=[1.0,0.5,0.1]; probs=[softmax(sims/t)[0] for t in temps]
plt.figure(figsize=(6,3)); plt.plot(temps,probs,'-o'); plt.gca().invert_xaxis(); plt.xlabel('temperature'); plt.ylabel('P(positive)'); plt.title('lower tau sharpens the decision')
assert probs[-1]>probs[0]

In [ ]:
# 3) masking creates a prediction problem from unlabeled audio
T=10; mask=np.zeros(T,dtype=bool); mask[[2,5,8]]=True; ratio=mask.mean()
plt.figure(figsize=(6,3)); plt.bar(np.arange(T),mask.astype(int)); plt.xlabel('time step'); plt.ylabel('masked'); plt.title(f'mask ratio={ratio:.3f}')
assert abs(ratio-0.3)<1e-9

In [ ]:
# 4) HuBERT-like masked cluster prediction uses cross-entropy
p=np.array([0.1,0.2,0.7]); target=2; ce=-math.log(p[target])
plt.figure(figsize=(6,3)); plt.bar([0,1,2],p); plt.scatter([target],[p[target]],c='r',zorder=5); plt.ylim(0,1); plt.title(f'cluster CE={ce:.3f}')
assert abs(ce-0.357)<0.001

In [ ]:
# 5) pretraining separates repeated acoustic units in embedding space
rng=np.random.default_rng(0); units=np.repeat([0,1,2],20); centers=np.array([[1,0],[0,1],[-1,0]]); emb=centers[units]+0.08*rng.normal(size=(len(units),2)); within=np.mean([np.linalg.norm(emb[units==u]-centers[u],axis=1).mean() for u in [0,1,2]]); between=np.linalg.norm(centers[0]-centers[1])
plt.figure(figsize=(4,4)); plt.scatter(emb[:,0],emb[:,1],c=units); plt.axis('equal'); plt.title('self-supervised units form clusters')
assert within<0.2 and between>1.0